In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ----------------------------------------------------
# SETTINGS
# ----------------------------------------------------

DEV_MODE = True
TEST_ROADS = [2, 3, 9]

LANE_WIDTH = 3.5

# ----------------------------------------------------
# FILE PATHS
# ----------------------------------------------------

try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

#project root directory (one level up from the codes directory)
project_root = codes_dir.parent

#------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

data_folder = project_root / "Data"

# ----------------------------------------------------
# LOAD DATA
# ----------------------------------------------------

# excel = pd.read_excel("")
# roads = gpd.read_file("")
excel = pd.read_excel(
    r"C:\School\TieDataProject\Vaylavirasto-data-projekti\Data\tiet-2-3-9.xlsx",
    header=0)

roads = gpd.read_file(
    r"C:\School\TieDataProject\KokoSuomi_Digiroad_R_GeoPackage\KokoSuomi_Digiroad_R_GeoPackage.gpkg",
    layer="DR_LINKKI",
    columns=["TIENUMERO", "TIEOSANRO", "AET", "LET", "geometry"]
)

# Normalize column names
excel = excel.rename(columns=str.lower)
roads = roads.rename(columns=str.lower)

roads = roads.rename(columns={
    "tienumero": "tie",
    "tieosanro": "aosa",
    "aet": "road_aet",
    "let": "road_let",
})

# ----------------------------------------------------
# DEV FILTER (OPTIONAL)
# ----------------------------------------------------

if DEV_MODE:
    excel = excel[excel["tie"].isin(TEST_ROADS)]
    roads = roads[roads["tie"].isin(TEST_ROADS)]

# ----------------------------------------------------
# FILTER ROADS TO ONLY NEEDED SEGMENTS
# ----------------------------------------------------

pairs = excel[["tie", "aosa"]].drop_duplicates()
roads = roads.merge(pairs, on=["tie", "aosa"])

# ----------------------------------------------------
# CLEAN + NORMALIZE EXCEL
# ----------------------------------------------------

excel_clean = excel.copy()

excel_clean["ajorata"] = excel_clean["ajorata"].fillna(0)
excel_clean["kaista"] = excel_clean["kaista"].fillna(11)

# ----------------------------------------------------
# VECTOR LANE LOGIC
# ----------------------------------------------------

excel_clean["ajorata"] = excel_clean["ajorata"].fillna(0)
excel_clean["kaista"] = excel_clean["kaista"].fillna(11)

# Direction
excel_clean["direction"] = 1

# ajorata = 0 → infer from kaista
mask0 = excel_clean["ajorata"] == 0
excel_clean.loc[mask0 & (excel_clean["kaista"] >= 20), "direction"] = -1

# ajorata overrides
excel_clean.loc[excel_clean["ajorata"] == 1, "direction"] = 1
excel_clean.loc[excel_clean["ajorata"] == 2, "direction"] = -1

# lane index PER DIRECTION
excel_clean["lane_index"] = excel_clean["kaista"] % 10
excel_clean.loc[excel_clean["lane_index"] == 0, "lane_index"] = 1

# normalize INSIDE each direction
excel_clean["lane_index"] = (
    excel_clean.groupby(["tie","aosa","direction"])["lane_index"]
    .rank(method="dense")
    .astype(int)
)

# ----------------------------------------------------
# REMOVE DUPLICATES AT SOURCE
# ----------------------------------------------------

excel_clean = excel_clean.sort_values(
    ["tie", "aosa", "direction", "lane_index", "aet"]
)

excel_clean = excel_clean.drop_duplicates(
    subset=["tie", "aosa", "direction", "lane_index"]
)

# ----------------------------------------------------
# SPLIT ROADS INTO TWO DIRECTIONS (CRITICAL)
# ----------------------------------------------------

roads_pos = roads.copy()
roads_pos["direction"] = 1

roads_neg = roads.copy()
roads_neg["direction"] = -1

roads = pd.concat([roads_pos, roads_neg], ignore_index=True)

# ----------------------------------------------------
# MERGE
# ----------------------------------------------------

merged = roads.merge(
    excel_clean[["tie","aosa","direction","lane_index"]],
    on=["tie","aosa","direction"],
    how="inner"
)

print(
    merged.groupby(["tie","aosa","direction"])["lane_index"]
    .nunique()
)

# ----------------------------------------------------
#  LANE COUNT (VECTOR)
# ----------------------------------------------------

lane_counts = (
    merged.groupby(["tie", "aosa", "direction"])["lane_index"]
    .max()
    .rename("lane_count")
)

merged = merged.join(lane_counts, on=["tie", "aosa", "direction"])

# ----------------------------------------------------
# COMPUTE OFFSET
# ----------------------------------------------------

center_shift = (merged["lane_count"] - 1) / 2
offset_index = merged["lane_index"] - 1 - center_shift

merged["offset"] = offset_index * LANE_WIDTH * merged["direction"]

# ----------------------------------------------------
# APPLY GEOMETRY OFFSET
# ----------------------------------------------------

def safe_offset(geom, offset):
    if geom is None or geom.is_empty:
        return geom
    try:
        return geom.parallel_offset(offset, "left", join_style=2)
    except Exception:
        return geom

merged["geometry"] = [
    safe_offset(g, o) for g, o in zip(merged.geometry, merged.offset)
]

# ----------------------------------------------------
#  FINAL CLEANUP
# ----------------------------------------------------

merged = merged[[
    "tie",
    "aosa",
    "direction",
    "lane_index",
    "lane_count",
    "geometry"
]].copy()

# ----------------------------------------------------
# EXPORT
# ----------------------------------------------------

merged.to_file(output_folder / "road_condition.gpkg", driver="GPKG")

print("Saved to:", output_folder)

# ----------------------------------------------------
# DEBUG CHECKS
# ----------------------------------------------------

print("\nMax rows per segment:")
print(merged.groupby(["tie", "aosa"]).size().max())

print("\nLane count distribution:")
print(
    merged.groupby(["tie", "aosa", "direction"])["lane_index"]
    .nunique()
    .value_counts()
)